# Summer NDVI at Colossus and Brownsville

Python conversion of the original NDVICode.txt for Google Colab. Uses the original QA60 cloud/cirrus mask, 5 km buffers, June–August 2022–2023 and 2025–2026 pooled medians, after-minus-before change, colour scales and six 10 m EPSG:4326 Google Drive exports. No cloud-mask substitution is applied.

Run from top to bottom and sign in through the Colab popup.


In [12]:
%pip -q install earthengine-api geemap

In [13]:
import ee
import geemap
from google.colab import output
from IPython.display import display
output.enable_custom_widget_manager()
PROJECT_ID = "ee-project-ntu-512407"
ee.Authenticate(auth_mode="colab")
ee.Initialize(project=PROJECT_ID)
print("Earth Engine connected:", ee.Number(1).getInfo())


Earth Engine connected: 1


## Settings
The six GeoTIFFs will be exported to the RemoteSensingProject folder in your Google Drive. Brownsville is a provisional comparison site.

In [14]:
RADIUS_M = 5000
EXPORT_FOLDER = "RemoteSensingProject"
SCALE = 10
EXPORT_CRS = "EPSG:4326"  # Preserves the original script's export CRS.
LOCATIONS = {
    "Site1_Memphis": (-90.15639, 35.06000),
    "Site2_Control": (-89.26083, 35.59056),
}
NDVI_VIS = {"min": 0, "max": .8, "palette": ["white", "lightgreen", "darkgreen"]}
CHANGE_VIS = {"min": -.3, "max": .3, "palette": ["red", "white", "blue"]}


In [15]:
def prepare(image):
    qa = image.select("QA60")
    mask = (
        qa.bitwiseAnd(1 << 10).eq(0)
        .And(qa.bitwiseAnd(1 << 11).eq(0))
    )

    masked = ee.Image(
        image.updateMask(mask)
        .divide(10000)
        .copyProperties(image, ["system:time_start"])
    )

    ndvi = masked.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return masked.addBands(ndvi)

results = {}
for name, (lon, lat) in LOCATIONS.items():
    roi = ee.Geometry.Point([lon, lat]).buffer(RADIUS_M)
    summer = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(roi).filter(ee.Filter.calendarRange(6, 8, "month")))
    composites = {}
    for label, start, end in [("before", "2022-01-01", "2024-01-01"),
                              ("after", "2025-01-01", "2027-01-01")]:
        collection = summer.filterDate(start, end)
        count = collection.size().getInfo()
        print(name, label, "candidate images:", count)
        if not count:
            raise ValueError(f"No candidate imagery for {name} {label}")
        composite = collection.map(prepare).select("NDVI").median().clip(roi)
        composites[label] = composite
    composites["change"] = composites["after"].subtract(composites["before"])
    results[name] = {"roi": roi, **composites}


Site1_Memphis before candidate images: 68
Site1_Memphis after candidate images: 92
Site2_Control before candidate images: 68
Site2_Control after candidate images: 90


## Interactive NDVI maps
Before and after layers are initially hidden, as in the original script. The difference layer is visible.


In [16]:
m = geemap.Map(
    center=[35.06000, -90.15639],
    zoom=12,
    basemap="SATELLITE"
)
for name, products in results.items():
    m.addLayer(products["before"], NDVI_VIS, name + " Pre-Ops NDVI (Summer 22-23)", False)
    m.addLayer(products["after"], NDVI_VIS, name + " Post-Ops NDVI (Summer 25-26)", False)
    m.addLayer(products["change"], CHANGE_VIS, name + " NDVI Difference")
display(m)


Map(center=[35.06, -90.15639], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

## Start the six Google Drive exports
Run once. Exports continue on Earth Engine in the background. Rerunning this cell in the same session skips already submitted tasks. A new session will not remember submitted tasks.

In [17]:
if "export_tasks" not in globals():
    export_tasks = {}
for name, products in results.items():
    for key, suffix in [("before", "2022_2023"), ("after", "2025_2026"), ("change", "Difference")]:
        description = f"{name}_NDVI_Summer_{suffix}"
        if description in export_tasks:
            print("Already submitted:", description)
            continue
        task = ee.batch.Export.image.toDrive(
            image=products[key], description=description,
            folder=EXPORT_FOLDER, fileNamePrefix=description,
            scale=SCALE, region=products["roi"], crs=EXPORT_CRS,
            maxPixels=1e10, fileFormat="GeoTIFF")
        task.start()
        export_tasks[description] = task
        print("Submitted:", description)


Already submitted: Site1_Memphis_NDVI_Summer_2022_2023
Already submitted: Site1_Memphis_NDVI_Summer_2025_2026
Already submitted: Site1_Memphis_NDVI_Summer_Difference
Already submitted: Site2_Control_NDVI_Summer_2022_2023
Already submitted: Site2_Control_NDVI_Summer_2025_2026
Already submitted: Site2_Control_NDVI_Summer_Difference


## Check export progress
Rerun this cell to refresh status. Once tasks show COMPLETED, open Google Drive → RemoteSensingProject. No Drive mounting is required. If authentication fails, resolve the Google sign-in before running the analysis; public imagery still requires an authenticated Earth Engine session.

In [18]:
for name, task in export_tasks.items():
    status = task.status()
    print(name, "—", status["state"])
    if status.get("error_message"):
        print("  Error:", status["error_message"])


Site1_Memphis_NDVI_Summer_2022_2023 — COMPLETED
Site1_Memphis_NDVI_Summer_2025_2026 — COMPLETED
Site1_Memphis_NDVI_Summer_Difference — COMPLETED
Site2_Control_NDVI_Summer_2022_2023 — COMPLETED
Site2_Control_NDVI_Summer_2025_2026 — COMPLETED
Site2_Control_NDVI_Summer_Difference — COMPLETED
